# Chart Insertion
This Jupyter notebook is used to insert new charts in the database. It works as follows:
1. Select the file.csv where the chart data is stored.
2. Select the chart type, category of the charts and the database in which to send them.
3. All informations about the variables is derived from a file.json.
4. New rows are generated according to different procedures (custom grouping is also an option).
5. New rows are inserted into the file.csv and the latter saved overwriting the old one.
6. Inspection of new variable groups to write a title and a description for the chart.
7. Update the new rows with both title and description.
8. Transform the title into semantic numerical vectors for subsequent semantic search.
9. Send the new rows with all the details.

In [12]:
import pandas as pd
import json
import os
import json
from toolbi import generate_chart_rows, update_chart_rows

In [13]:
# ------- Chart file info
file_path= "/Users/gabrielvincenzi/Desktop/Divolgo/Sunflowerpy/charts/chartsData.csv"
columns = ["id", "title", "description", "db_name", "category", "chart_type", "vars", "vector_dim"]
if os.path.exists(file_path):
    try:
        df = pd.read_csv(file_path)
        # If it has no columns (completely empty), reset it
        if df.empty and len(df.columns) == 0:
            df = pd.DataFrame(columns=columns)
            df.to_csv(file_path, index=False)
    except pd.errors.EmptyDataError:
        df = pd.DataFrame(columns=columns)
        df.to_csv(file_path, index=False)
else:
    df = pd.DataFrame(columns=columns)
    df.to_csv(file_path, index=False)

# ------- Automatically determine the next id
if df.empty:
    next_id = 1
else:
    next_id = int(df["id"].max() + 1)

In [18]:
# ------- Selected constants for these charts
chart_type = "hist"
category = "earn"
db_name = "e_earnings"

with open('output.json', 'r') as file:
    data = json.load(file)

In [ ]:
def generate_chart_rows(
    data: Dict[str, Any],
    db_name: str,
    procedure: int = None,
    forbidden_group: set = None,
    forbidden_other: set = set(),
    delete_single_var_rows: bool = False,
    start_id: int = 1,
    chart_type: str = "",
    category: str = "",
    grouping: Optional[Dict[str, List[str]]] = None,
) -> List[Dict[str, Any]]:
    """
    Generate rows using procedures from the provided `data` dict for the given `db_name`.

    Parameters:
    - data: json file with data to be parsed.
    - db_name: name of the database (present in the first level of the json file) to be selected.
    - procedure: one of the following procedures to create groupings of variables and attributes:
        - 1: Graph for a unique Root bucket and a unique Selection.
        - 2: Graph for a unique Variable and each attribute with each other selection.
        - 3: Graph for a unique Root bucket and all combinations of selections in all attributes (Cartesian product).
    - delete_single_var_rows: it is possible that the procedures create singular lists, if this option is set to
      true, then these rows are not appended.
    - start_id: numeric id from which the rows start to be indexed, it increments only for rows included in the returned list
    - chart_type: the type of chart (hist, line, radar, area, pie) to be used.
    - category: the category of the chart.
    - grouping (optional): dict mapping root name -> list of top-level keys (those present in data[db_name]).
      If provided, this explicit grouping is used. Otherwise grouping is inferred from each top-level key
      by splitting at the first '-' and using the left part as the root (original behaviour).
    - forbidden_group: in procedure 2 these attributes are never going to be used as the fixed list for combinations, 
      while in procedure 3 these attributes will create n different lists filled with Cartesian product of their selections with all other selections.
    - forbidden_other: in procedure 2 these attributes are never going to be used as the changable list for combinations.

    Structure json:
      variable: {attribute1: {selection1, selection2}, attribute2: {selection1, selection2}}
      variable = root + name
    """
    if db_name not in data:
        raise KeyError(f"db_name '{db_name}' not found in data")

    rows: List[Dict[str, Any]] = []
    next_id = int(start_id)

    # Helper: build objects dict for a given list of keys (only keep existing keys)
    def build_objs_for_keys(keys: List[str]) -> Dict[str, Dict[str, List[str]]]:
        objs: Dict[str, Dict[str, List[str]]] = {}
        for key in keys:
            if key not in data[db_name]:
                raise KeyError(f"key '{key}' from grouping not found in data['{db_name}']")
            value = data[db_name][key]
            objs[key] = {}
            for attr, val in value.items():
                if attr == "description":
                    continue
                # ensure string -> split by comma; keep deterministic ordering
                vals = [x.strip() for x in str(val).split(",") if x.strip()]
                objs[key][attr] = vals
        return objs
    
    # Helper to build vars_list for a given mapping of attribute -> chosen value(s)
    def build_vars_list_for_fixed_map(fixed_map: Dict[str, str]) -> List[str]:
        """
        fixed_map: mapping for some attributes (usually the primary attrs). For
        the remaining attributes, iterate all combinations (lexicographic).
        Return a single aggregated vars list (one element per variable name).
        """
        remaining_attrs = [a for a in attr_names if a not in fixed_map]
        remaining_lists = [attr_unique[a] for a in remaining_attrs]

        remaining_combos = list(product(*remaining_lists)) if remaining_lists else [()]

        vars_list: List[str] = []
        # For each remaining-combination (lexicographic), append variables for each object
        for rem_combo in remaining_combos:
            rem_map = dict(zip(remaining_attrs, rem_combo)) if remaining_attrs else {}
            full_map = {**fixed_map, **rem_map}
            for obj_name in objs:
                parts = [obj_name] + [str(full_map[attr]) for attr in attr_names]
                vars_list.append("_".join(parts))
        return vars_list

    if procedure == 1:
        # Build groups per root either from provided grouping or by splitting keys
        groups: Dict[str, Dict[str, Dict[str, List[str]]]] = {}

        if grouping is not None:
            # Validate and build groups according to provided mapping
            # Expect mapping: root -> list of top-level keys
            for root, keys in grouping.items():
                if not isinstance(keys, (list, tuple)):
                    raise TypeError(f"grouping['{root}'] must be a list of keys")
                groups[root] = build_objs_for_keys(list(keys))
        else:
            # Infer grouping by prefix before '-'
            # groups[root][key] = { attr: [vals...] }
            for key, value in data[db_name].items():
                root = key.split('-', 1)[0]
                groups.setdefault(root, {})
                groups[root].setdefault(key, {})
                for attr, val in value.items():
                    if attr == "description":
                        continue
                    groups[root][key][attr] = [x.strip() for x in str(val).split(",") if x.strip()]

        # Iterate per root
        for root, objs in groups.items():
            if not objs:
                continue

            # Collect attribute names from the first object (assume consistent attrs)
            first_obj = next(iter(objs.values()))
            attr_names = list(first_obj.keys())

            # For each attribute, collect unique values across all objects in this root
            attr_unique: Dict[str, List[str]] = {}
            for attr in attr_names:
                vals = set()
                for obj in objs.values():
                    vals.update(obj.get(attr, []))
                # deterministic order
                attr_unique[attr] = sorted(vals)

            # Build Cartesian product of attributes
            combos = list(product(*[attr_unique[attr] for attr in attr_names]))

            for combo in combos:
                combo_dict = dict(zip(attr_names, combo))
                vars_list = []
                for obj_name in objs:
                    var_parts = [obj_name] + [combo_dict[attr] for attr in attr_names]
                    vars_list.append("_".join(var_parts))
                    
                vars_string = "+".join(vars_list)
                row = {
                    "id": next_id,
                    "title": "",
                    "description": "",
                    "db_name": db_name,
                    "vars": vars_string,
                    "chart_type": chart_type,
                    "category": category,
                    "vector_dim": "",
                }
                if not (delete_single_var_rows and len(vars_list) == 1):
                    rows.append(row)
                    next_id += 1

    elif procedure == 2:
        for key, value in data[db_name].items():
            attributes: Dict[str, List[str]] = {}
            for attribute, val in value.items():
                if attribute == "description":
                    continue
                attr_list = [x.strip() for x in str(val).split(",") if x.strip()]
                attributes[attribute] = attr_list

            if not attributes:
                continue

            attr_names = list(attributes.keys())  # preserve original order
            n_attrs = len(attr_names)

            # configuration: never use these as the varying/group attribute
            if not forbidden_group:
                forbidden_group = {"unit"}
            
            preferred_placeholders = ["total", "tot", "t"]

            # Determine placeholder for each forbidden_other attribute based on existing values
            forbidden_placeholder_map: Dict[str, str] = {}
            for forb in forbidden_other:
                # find the attribute in attr_names (case-insensitive)
                matched_attr_name = next((an for an in attr_names if an.lower() == forb.lower()), None)
                if matched_attr_name is None:
                    continue  # attribute not present, skip

                values_lower = [str(v).lower() for v in attributes[matched_attr_name]]
                # pick the first preferred placeholder present in the values
                chosen = next((ph for ph in preferred_placeholders if ph.lower() in values_lower), None)
                if chosen is None:
                    # If none of the preferred placeholders present, skip this attribute
                    continue
                forbidden_placeholder_map[matched_attr_name.lower()] = chosen

            # Generate combinations
            for g in range(n_attrs):
                group_name = attr_names[g]

                # Skip disallowed group attributes
                if group_name.lower() in {x.lower() for x in forbidden_group}:
                    continue

                group_list = attributes[group_name]

                # other indices: exclude group AND forbidden_other
                other_indices = [
                    i for i in range(n_attrs)
                    if i != g and attr_names[i].lower() not in {x.lower() for x in forbidden_other}
                ]
                other_lists = [attributes[attr_names[i]] for i in other_indices]

                other_combinations = list(product(*other_lists)) if other_lists else [()]

                for combo in other_combinations:
                    vars_list = []
                    for group_item in group_list:
                        parts = []
                        combo_idx = 0
                        for i in range(n_attrs):
                            name_lower = attr_names[i].lower()
                            if i == g:
                                parts.append(str(group_item))
                            elif name_lower in forbidden_placeholder_map:
                                # use the determined placeholder (must exist in values)
                                parts.append(forbidden_placeholder_map[name_lower])
                            else:
                                parts.append(str(combo[combo_idx]))
                                combo_idx += 1

                        suffix = "_".join(parts)
                        vars_list.append(f"{key}_{suffix}")

                    vars_string = "+".join(vars_list)
                    row = {
                        "id": next_id,
                        "title": "",
                        "description": "",
                        "db_name": db_name,
                        "vars": vars_string,
                        "chart_type": chart_type,
                        "category": category,
                        "vector_dim": "",
                    }
                    if not (delete_single_var_rows and len(vars_list) == 1):
                        rows.append(row)
                    next_id += 1

    elif procedure == 3:
        # Build groups per root (same grouping logic as proc 1)
        groups: Dict[str, Dict[str, Dict[str, List[str]]]] = {}

        if grouping is not None:
            for root, keys in grouping.items():
                if not isinstance(keys, (list, tuple)):
                    raise TypeError(f"grouping['{root}'] must be a list of keys")
                groups[root] = build_objs_for_keys(list(keys))
        else:
            for key, value in data[db_name].items():
                root = key.split('-', 1)[0]
                groups.setdefault(root, {})
                groups[root].setdefault(key, {})
                for attr, val in value.items():
                    if attr == "description":
                        continue
                    groups[root][key][attr] = [x.strip() for x in str(val).split(",") if x.strip()]

        if not forbidden_group:
            forbidden_group = {"unit"}

        forbidden_primary_lc = {x.lower() for x in (forbidden_group or set())}

        for root, objs in groups.items():
            if not objs:
                continue

            # attribute order is taken from the first object (preserve keys order)
            first_obj = next(iter(objs.values()))
            attr_names = list(first_obj.keys())

            # collect unique sorted values per attribute across objects in the root
            attr_unique: Dict[str, List[str]] = {}
            for attr in attr_names:
                vals = set()
                for obj in objs.values():
                    vals.update(obj.get(attr, []))
                attr_unique[attr] = sorted(vals)

            # determine primary attributes (to split on)
            primary_attrs = [a for a in attr_names if a.lower() in forbidden_primary_lc]

            if not primary_attrs:
                # No forbidden_primary present => single aggregated var-list for the entire cartesian product
                # Build one fixed_map == {} so remaining are all attributes
                vars_list = build_vars_list_for_fixed_map({})
                if not (delete_single_var_rows and len(vars_list) == 1):
                    rows.append({
                        "id": next_id,
                        "title": "",
                        "description": "",
                        "db_name": db_name,
                        "vars": "+".join(vars_list),
                        "chart_type": chart_type,
                        "category": category,
                        "vector_dim": "",
                    })
                    next_id += 1
            else:
                # Split output by the cartesian product of primary attribute values
                primary_value_lists = [attr_unique[a] for a in primary_attrs]
                # Skip root if any primary attribute has no values
                if any(len(lst) == 0 for lst in primary_value_lists):
                    continue

                for prim_values in product(*primary_value_lists):
                    fixed_primary_map = dict(zip(primary_attrs, prim_values))
                    vars_list = build_vars_list_for_fixed_map(fixed_primary_map)
                    if not (delete_single_var_rows and len(vars_list) == 1):
                        rows.append({
                            "id": next_id,
                            "title": "",
                            "description": "",
                            "db_name": db_name,
                            "vars": "+".join(vars_list),
                            "chart_type": chart_type,
                            "category": category,
                            "vector_dim": "",
                        })
                        next_id += 1

    else:
        raise ValueError("Procedure not recognized")

    return rows


In [31]:
rows_out = generate_chart_rows(
    data, 
    db_name, 
    procedure=3, 
    delete_single_var_rows=True, 
    start_id=next_id, 
    chart_type=chart_type, 
    category=category,
    forbidden_group={"sex"}
)

df_out = pd.concat([df, pd.DataFrame(rows_out)], ignore_index=True)
#df_out.to_csv(file_path, index=False)

In [ ]:
updates = [
    ["", ""],
    ["", ""],
    ["", ""],
    ["", ""],
    ["", ""],
    ["", ""],
    ["", ""],
    ["", ""],
    ["", ""],
    ["", ""],
    ["", ""],
    ["", ""],
    ["", ""],
    ["", ""],
    ["", ""],
    ["", ""],
    ["", ""],
    ["", ""],
    ["", ""],
    ["", ""],
    ["", ""],
]

In [18]:
df_updated = update_chart_rows(df_out, updates, id_val=next_id, send=False, full_update=False, dest_table="charts")
#df_updated.to_csv(file_path, index=False)

ValueError: Mismatch: 45 rows but 2 updates